In [1]:
import json, re
from collections import defaultdict, Counter

In [2]:
docs = json.load(open("../data/docs.json", encoding="utf-8"))
chunks = json.load(open("../data/chunks_full.json", encoding="utf-8"))
titles = {c["title"] for c in chunks if c["kind"]=="api"}

In [5]:
tail_index = defaultdict(list)
for t in titles:
    tail_index[t.split(".")[-1]].append(t)

REF = re.compile(r":(func|class|meth|mod|attr):`([^`]+)`")

In [6]:
def clean_ref(raw):
    m = re.search(r"<([^>]+)>", raw)
    if m:
        raw = m.group(1)
    raw = raw.strip()
    raw = raw.lstrip("~.")
    if raw.endswith("()"):
        raw = raw[:-2]
    return raw

In [24]:
def resolve(src, name):
    if name in titles:
        return name
    parts = src.split(".")
    cand = ".".join(parts[:-1] + [name])        # сосед в том же классе или модуле
    if cand in titles:
        return cand
    if "torch." + name in titles:
        return "torch." + name
    cand = ".".join(parts[:-2] + [name])        # на уровень выше
    if cand in titles:
        return cand
    if "." not in name:
        c = tail_index.get(name, [])
        if len(c) == 1:
            return c[0]
    return None

In [13]:
node_ids = defaultdict(list)
for c in chunks:
    if c["kind"]=="api":
        node_ids[c["title"]].append(c["id"])
    else:
        last = c["title"].split(" > ")[-1].strip()
        if re.fullmatch(r"torch(\.\w+)+", last):
            node_ids[last].append(c["id"])

titles = set(node_ids)
tail_index = defaultdict(list)
for t in titles:
    tail_index[t.split(".")[-1]].append(t)

In [14]:
by_role = defaultdict(lambda: [0, 0])
edges, unresolved = set(), Counter()

for src, doc in docs.items():
    for role, raw in REF.findall(doc):
        if role == "attr":
            continue
        name = clean_ref(raw)
        dst = resolve(src, name)
        by_role[role][0] += 1
        if dst:
            by_role[role][1] += 1
            if dst != src:
                edges.add((src, dst))
        else:
            unresolved[name] += 1

for role, (total, ok) in by_role.items():
    print(role, total, "allowed", ok, f"({ok / total:.0%})")
print("unique edges: ", len(edges))

class 1924 allowed 1506 (78%)
meth 457 allowed 308 (67%)
func 1888 allowed 1680 (89%)
mod 10 allowed 3 (30%)
unique edges:  2561


In [15]:
json.dump({"edges": sorted(edges), "nodes": dict(node_ids)}, open("../data/graph.json", "w", encoding="utf-8"), ensure_ascii=False)
print(len(edges), len(node_ids))

2561 3577


In [16]:
edges_out = defaultdict(list)
indeg = Counter()
for a, b in sorted(edges):
    edges_out[a].append(b)
    indeg[b] += 1

def node_name(c):
    if c["kind"] == "api":
        return c["title"]
    else:
        return c["title"].split(" > ")[-1].strip()

def expand(res, top=3, per_chunk=2, max_indeg=20):
    ids = {c["id"] for s, c in res}
    extra = []
    for s, c in res[:top]:
        added = 0
        for dst in edges_out.get(node_name(c), []):
            if added >= per_chunk:
                break
            if indeg[dst] > max_indeg:
                continue
            for cid in node_ids[dst]:
                if cid not in ids:
                    extra.append((0.0, chunks[cid]))
                    ids.add(cid)
                    added += 1
                    break

    return res + extra

In [18]:
import sys
sys.path.insert(0, "..")  # rag.py лежит в корне проекта
from rag import RAG, is_hit

rag = RAG()
reason, res = rag.retrieve("what does torch.nn.DataParallel do")
print(reason)
for s, c in res:
    print(f"{s:.2f}", c["title"][:80])

None
1.00 torch.nn.DataParallel
1.00 torch.nn.DataParallel
0.66 Distributed Data Parallel
0.66 torch.nn.parallel.DistributedDataParallel
0.64 torch.distributed.tensor.parallel.SequenceParallel


In [19]:
for s, c in expand(res):
    print(f"{s:.2f}", c["title"][:80])

1.00 torch.nn.DataParallel
1.00 torch.nn.DataParallel
0.66 Distributed Data Parallel
0.66 torch.nn.parallel.DistributedDataParallel
0.64 torch.distributed.tensor.parallel.SequenceParallel
0.00 torch.autograd.Function.forward
0.00 torch.nn.BatchNorm2d
0.00 torch.nn.BatchNorm2d
0.00 torch.nn.parallel.DistributedDataParallel


In [20]:
with open("../data/eval_graph.json", encoding="utf-8") as f:
    graph_items = json.load(f)

def covered(it, res):
    got = sum(any(is_hit(c["title"], [g]) for s, c in res) for g in it["gold"])
    return got / len(it["gold"])


base = grafted = 0.0
helped, eligible, refused = [], 0, 0
for it in graph_items:
    reason, res = rag.retrieve(it["q"])
    if reason:
        refused += 1
        print("ОТКАЗ:", it["q"], "|", reason)
        continue
    res2 = expand(res)
    b, a = covered(it, res), covered(it, res2)
    base += b
    grafted += a
    src_in = any(is_hit(c["title"], it["src"]) for s, c in res)
    eligible += src_in and b < 1
    if a > b:
        helped.append(it["q"])
n = len(graph_items) - refused
print("вопросов:", n, "| отказов:", refused)
print("покрытие gold без графа:", round(base / n, 2), "| с графом:", round(grafted / n, 2))
print("граф мог помочь (исходный чанк найден, gold неполный):", eligible, "| помог:", len(helped))
for q in helped:
    print("  помог:", q)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

вопросов: 14 | отказов: 0
покрытие gold без графа: 0.25 | с графом: 0.68
граф мог помочь (исходный чанк найден, gold неполный): 9 | помог: 8
  помог: what is a less experimental and faster alternative to torch.autograd.functional.jacobian
  помог: what should I use instead of Tensor.resize_ to change the shape of a tensor
  помог: какие функции использовать для нормы вектора и нормы матрицы вместо torch.norm
  помог: what is torch.linalg.matmul an alias of
  помог: which two operations is CrossEntropyLoss equivalent to when the input has class indices
  помог: что делать, если view нельзя выполнить для тензора с такой раскладкой
  помог: which function samples from the same distribution as Categorical
  помог: is transpose_copy the same as transpose, and how does the output differ


In [21]:
CUES = [
    ("use_instead", r"instead|recommend|prefer|should use|consider using"),
    ("see_also",    r"see also"),
    ("alias",       r"alias for|alias of"),
    ("equivalent",  r"equivalent|same as|same operation"),
    ("deprecated",  r"deprecated|superseded|replaced"),
]
SENT = re.compile(r"(?<=[.!?])\s+|\n\n")

def edge_types(src, doc):
    out = {}
    for sent in SENT.split(doc):
        rel = next((name for name, pat in CUES if re.search(pat, sent, re.I)), None)
        if not rel:
            continue
        for role, raw in REF.findall(sent):
            if role == "attr":
                continue
            name = clean_ref(raw)
            dst = resolve(src, name)
            if dst and dst not in out and dst != src:
                out[dst] = rel
    return out


In [22]:
for it in graph_items:
    reason, res = rag.retrieve(it["q"])
    res2 = expand(res)
    src_in = any(is_hit(c["title"], it["src"]) for s, c in res)
    b, a = covered(it, res), covered(it, res2)
    if not src_in:
        print("ИСХОДНЫЙ НЕ НАЙДЕН:", it["q"])
    elif a < 1:
        print("НЕ ДОБРАЛ:", it["q"], "| gold:", it["gold"], "| покрытие", round(a, 2))

НЕ ДОБРАЛ: what is a less experimental and faster alternative to torch.autograd.functional.jacobian | gold: ['torch.func.jacfwd', 'torch.func.jacrev'] | покрытие 0.5
НЕ ДОБРАЛ: what should I use instead of Tensor.resize_ to change the shape of a tensor | gold: ['torch.Tensor.view', 'torch.Tensor.reshape'] | покрытие 0.5
ИСХОДНЫЙ НЕ НАЙДЕН: how to broadcast tensors between processes when you do not need to send Python objects
НЕ ДОБРАЛ: чему равна функция greater_equal | gold: ['torch.ge'] | покрытие 0.0
НЕ ДОБРАЛ: что делать, если view нельзя выполнить для тензора с такой раскладкой | gold: ['torch.Tensor.reshape', 'torch.Tensor.contiguous'] | покрытие 0.5
ИСХОДНЫЙ НЕ НАЙДЕН: how do I register my own custom operator


In [23]:
for it in graph_items:
    reason, res = rag.retrieve(it["q"])
    pos = next((i for i, (s, c) in enumerate(res) if is_hit(c["title"], it["src"])), None)
    if pos is None or covered(it, expand(res)) >= 1:
        continue
    print("Q:", it["q"])
    print("   место источника в выдаче:", pos, "| рёбра из источника:", len(edges_out.get(it["src"][0], [])))
    for g in it["gold"]:
        print("   ", g, "| есть ребро:", (it["src"][0], g) in edges, "| входящих:", indeg.get(g, 0))

Q: what is a less experimental and faster alternative to torch.autograd.functional.jacobian
   место источника в выдаче: 0 | рёбра из источника: 3
    torch.func.jacfwd | есть ребро: True | входящих: 2
    torch.func.jacrev | есть ребро: True | входящих: 3
Q: what should I use instead of Tensor.resize_ to change the shape of a tensor
   место источника в выдаче: 1 | рёбра из источника: 5
    torch.Tensor.view | есть ребро: True | входящих: 7
    torch.Tensor.reshape | есть ребро: True | входящих: 1
Q: чему равна функция greater_equal
   место источника в выдаче: 3 | рёбра из источника: 1
    torch.ge | есть ребро: True | входящих: 2
Q: что делать, если view нельзя выполнить для тензора с такой раскладкой
   место источника в выдаче: 1 | рёбра из источника: 5
    torch.Tensor.reshape | есть ребро: False | входящих: 1
    torch.Tensor.contiguous | есть ребро: True | входящих: 2


In [26]:
with open("../data/eval.json", encoding="utf-8") as f:
    items = json.load(f)
main_items = [it for it in items if it["gold"]]      # 34 вопроса с ответом
print(len(main_items), len(graph_items))

34 14


In [28]:
cache = {}
for it in graph_items + main_items:
    cache[it["q"]] = rag.retrieve(it["q"])
print(len(cache))

48


In [29]:
def run(top, per):
    cov, n, in_ctx, extra = 0.0, 0, 0, []
    for it in graph_items:
        reason, res = cache[it["q"]]
        if reason:
            continue
        cov += covered(it, expand(res, top=top, per_chunk=per))
        n += 1
    for it in main_items:
        reason, res = cache[it["q"]]
        if reason:
            continue
        res2 = expand(res, top=top, per_chunk=per)
        in_ctx += any(is_hit(c["title"], it["gold"]) for s, c in res2)
        extra.append(len(res2) - len(res))
    return cov / n, in_ctx, sum(extra) / len(extra)


for top, per in [(0, 0), (3, 2), (5, 2), (5, 3), (5, 4), (5, 6)]:
    cov, in_ctx, extra = run(top, per)
    print(f"top={top} per_chunk={per}: покрытие graph {cov:.2f} | eval.json gold в контексте {in_ctx}/{len(main_items)} | добавляется на вопрос {extra:.1f}")

top=0 per_chunk=0: покрытие graph 0.25 | eval.json gold в контексте 33/34 | добавляется на вопрос 0.0
top=3 per_chunk=2: покрытие graph 0.68 | eval.json gold в контексте 33/34 | добавляется на вопрос 0.9
top=5 per_chunk=2: покрытие graph 0.79 | eval.json gold в контексте 33/34 | добавляется на вопрос 1.6
top=5 per_chunk=3: покрытие graph 0.82 | eval.json gold в контексте 33/34 | добавляется на вопрос 1.9
top=5 per_chunk=4: покрытие graph 0.82 | eval.json gold в контексте 33/34 | добавляется на вопрос 2.0
top=5 per_chunk=6: покрытие graph 0.82 | eval.json gold в контексте 33/34 | добавляется на вопрос 2.1
